In [0]:
%sql
CREATE CATALOG IF NOT EXISTS telecom_catalog_assign;
CREATE SCHEMA IF NOT EXISTS telecom_catalog_assign.landing_zone;
CREATE VOLUME IF NOT EXISTS telecom_catalog_assign.landing_zone.landing_vol;


In [0]:
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/")


In [0]:
customer_csv = """101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
"""

usage_tsv = """customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
"""

tower_logs_region1 = """event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
"""


In [0]:
base_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"

dbutils.fs.put(f"{base_path}/customer/customer.csv", customer_csv, overwrite=True)
dbutils.fs.put(f"{base_path}/usage/usage.tsv", usage_tsv, overwrite=True)
dbutils.fs.put(f"{base_path}/tower/tower_region1.log", tower_logs_region1, overwrite=True)


In [0]:
df_customer = (
    spark.read
    .option("inferSchema", True)
    .csv(f"{base_path}/customer/customer.csv")
    .toDF("customer_id", "name", "age", "city", "plan_type")
)


In [0]:
df_usage = (
    spark.read
    .option("header", True)
    .option("delimiter", "\t")
    .option("inferSchema", True)
    .csv(f"{base_path}/usage/usage.tsv")
)


In [0]:
df_tower = (
    spark.read
    .option("header", True)
    .option("delimiter", "|")
    .option("inferSchema", True)
    .csv(f"{base_path}/tower/tower_region1.log")
)


In [0]:
from pyspark.sql.functions import col, when

df_customer_clean = (
    df_customer
    .withColumn("age", col("age").cast("int"))
    .withColumn("name", when(col("name").isNull(), "UNKNOWN").otherwise(col("name")))
)


In [0]:
base_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"


In [0]:
dbutils.fs.cp(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",
    f"{base_path}/customer/customer.csv",
    recurse=False
)


In [0]:
dbutils.fs.cp(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv",
    f"{base_path}/usage/usage.tsv",
    recurse=False
)


In [0]:
dbutils.fs.mkdirs(f"{base_path}/tower/region1/")
dbutils.fs.mkdirs(f"{base_path}/tower/region2/")


In [0]:
dbutils.fs.cp(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_region1.log",
    f"{base_path}/tower/region1/tower_region1.log",
    recurse=False
)


In [0]:
dbutils.fs.ls(f"{base_path}/customer/")
dbutils.fs.ls(f"{base_path}/usage/")
dbutils.fs.ls(f"{base_path}/tower/region1/")
dbutils.fs.ls(f"{base_path}/tower/region2/")


In [0]:
dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol")


In [0]:
base_tower_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower"


In [0]:
df_glob = (
    spark.read
    .option("header", True)
    .option("delimiter", "|")
    .option("pathGlobFilter", "*.log")
    .csv(f"{base_tower_path}/region1/")
)

df_glob.show()


In [0]:
df_multi_path = (
    spark.read
    .option("header", True)
    .option("delimiter", "|")
    .csv([
        f"{base_tower_path}/region1/",
        f"{base_tower_path}/region2/"
    ])
)

df_multi_path.show()


In [0]:
df_recursive = (
    spark.read
    .option("header", True)
    .option("delimiter", "|")
    .option("recursiveFileLookup", "true")
    .csv(base_tower_path)
)

df_recursive.show()


In [0]:
customer_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv"
usage_path    = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv"


In [0]:
df_cust_default = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "false")
    .csv(customer_path)
)

df_cust_default.printSchema()
df_cust_default.show()


In [0]:
df_usage_default = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "false")
    .option("delimiter", "\t")
    .csv(usage_path)
)

df_usage_default.printSchema()
df_usage_default.show()


In [0]:
df_cust_infer = (
    spark.read
    .option("header", "false")   # Customer file has no header
    .option("inferSchema", "true")
    .csv(customer_path)
    .toDF("customer_id", "name", "age", "city", "plan_type")
)

df_cust_infer.printSchema()
df_cust_infer.show()


In [0]:
df_usage_infer = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", "\t")
    .csv(usage_path)
)

df_usage_infer.printSchema()
df_usage_infer.show()


In [0]:
customer_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv"
usage_path    = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv"
tower_path    = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/"


In [0]:
df_customer = (
    spark.read
    .option("inferSchema", True)
    .csv(customer_path)
    .toDF("customer_id", "name", "age", "city", "plan_type")
)

df_customer.printSchema()
df_customer.show()


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

usage_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("voice_mins", IntegerType(), True),
    StructField("data_mb", IntegerType(), True),
    StructField("sms_count", IntegerType(), True)
])

df_usage = (
    spark.read
    .option("header", True)
    .option("delimiter", "\t")
    .schema(usage_schema)
    .csv(usage_path)
)

df_usage.printSchema()
df_usage.show()


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, TimestampType
)

tower_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("tower_id", StringType(), True),
    StructField("signal_strength", IntegerType(), True),
    StructField("timestamp", TimestampType(), True)
])

df_tower = (
    spark.read
    .option("header", True)
    .option("delimiter", "|")
    .schema(tower_schema)
    .csv(tower_path)
)

df_tower.printSchema()
df_tower.show()
